# **STEP 1: INSTALL DEPENDENCIES**

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.44.2
!pip install ultralytics einops timm pillow kagglehub
!pip install seaborn scikit-learn accelerate bitsandbytes

# **STEP 2: IMPORTS**

In [ ]:
import gc
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageDraw
import json
import random
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO
import kagglehub
import pandas as pd
import shutil
from scipy.spatial import ConvexHull
from skimage.measure import find_contours, perimeter

print("✅ All libraries imported successfully")

# **STEP 3: MEMORY MANAGEMENT**

In [ ]:
def clear_memory():
    """Clear GPU and CPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

clear_memory()
print(f"✅ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# **STEP 4: DATASET CONFIGURATION**

In [ ]:
CLASS_NAMES = ['Glass', 'Metal', 'Paper', 'Plastic']
NUM_CLASSES = 4

print(f"✅ Classes: {CLASS_NAMES}")
print(f"✅ Number of classes: {NUM_CLASSES}")

# **STEP 5: DOWNLOAD AND SETUP DATASET**

In [ ]:
def setup_dataset():
    """Download and setup BUU-WOD dataset"""

    data_path = Path("/content/BUU-WOD")

    if data_path.exists():
        print(f"✅ Dataset already exists at: {data_path}")
        return data_path

    try:
        print("📥 Downloading dataset from Kaggle...")
        path = kagglehub.dataset_download("visionlab1buu/buu-waste-occlusion-dataset")
        shutil.copytree(path, data_path, dirs_exist_ok=True)
        print(f"✅ Dataset downloaded to: {data_path}")
        return data_path
    except Exception as e:
        print(f"❌ Download error: {e}")
        print("⚠️ Please upload dataset manually to /content/BUU-WOD")
        return None

data_path = setup_dataset()

if data_path is None:
    print("❌ Dataset not found. Exiting...")
    exit()

# **STEP 6: VERIFY DATASET STRUCTURE**

In [ ]:
print("\n📁 Dataset Structure:")
for item in sorted(data_path.iterdir()):
    if item.is_dir():
        print(f"  📁 {item.name}")
        for sub in sorted(item.iterdir()):
            if sub.is_dir():
                files = len(list(sub.glob('*'))) if sub.exists() else 0
                print(f"    📁 {sub.name} ({files} files)")

# Check directories
train_img_dir = data_path / "1_Model_Training_Data" / "train" / "images"
train_label_dir = data_path / "1_Model_Training_Data" / "train" / "labels"
valid_img_dir = data_path / "1_Model_Training_Data" / "valid" / "images"
test_img_dir = data_path / "1_Model_Training_Data" / "test" / "images"

print(f"\n📊 Dataset Statistics:")
print(f"  Train images: {len(list(train_img_dir.glob('*')))}")
print(f"  Train labels: {len(list(train_label_dir.glob('*')))}")
print(f"  Valid images: {len(list(valid_img_dir.glob('*')))}")
print(f"  Test images: {len(list(test_img_dir.glob('*')))}")

# **STEP 7: CREATE/CORRECT data.yaml**

In [ ]:
yaml_path = data_path / "1_Model_Training_Data" / "data.yaml"

if yaml_path.exists():
    print(f"✅ data.yaml exists at: {yaml_path}")
else:
    print("⚠️ Creating data.yaml...")
    yaml_content = f"""
path: {data_path}/1_Model_Training_Data
train: train/images
val: valid/images
test: test/images

nc: {NUM_CLASSES}
names: {CLASS_NAMES}
"""
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    print("✅ data.yaml created")

# **STEP 8: TRAIN YOLOv8s-seg**

In [ ]:
def train_yolo():
    """Train YOLOv8s-seg on BUU-WOD dataset"""

    print("\n🚀 Starting YOLOv8s-seg training...")
    print(f"📁 Training on: {len(list(train_img_dir.glob('*')))} images")
    print(f"📁 Validating on: {len(list(valid_img_dir.glob('*')))} images")

    model = YOLO("yolov8s-seg.pt")

    results = model.train(
        data=str(yaml_path),
        epochs=30,
        batch=16,
        imgsz=640,
        device=0,
        workers=2,
        patience=10,
        project='waste_segmentation',
        name='yolov8s_seg',
        exist_ok=True,
        cache=False,

        # Hyperparameters
        lr0=0.001,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,

        # Augmentation for occlusion
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=10.0,
        translate=0.1,
        scale=0.5,
        shear=2.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.5,
        copy_paste=0.3,

        label_smoothing=0.1,
        overlap_mask=True,
    )

    print("✅ Training completed!")
    return model

# Train model
model = train_yolo()

# **STEP 9: LOAD BEST MODEL**

In [ ]:
best_model_path = Path('/content/waste_segmentation/yolov8s_seg/weights/best.pt')
if best_model_path.exists():
    model = YOLO(str(best_model_path))
    print(f"✅ Best model loaded from: {best_model_path}")
    print(f"✅ Model size: {best_model_path.stat().st_size / 1e6:.1f} MB")
else:
    print("⚠️ Best model not found. Using last model.")
    last_model_path = Path('/content/waste_segmentation/yolov8s_seg/weights/last.pt')
    if last_model_path.exists():
        model = YOLO(str(last_model_path))
        print(f"✅ Last model loaded from: {last_model_path}")

# **STEP 10: EVALUATE MODEL**

In [ ]:
def evaluate_model():
    """Evaluate trained model on test set"""

    print("\n📊 Evaluating model on test set...")

    try:
        metrics = model.val(
            data=str(yaml_path),
            split='test',
            conf=0.25,
            iou=0.45
        )

        print("\n" + "="*60)
        print("EVALUATION RESULTS ON BUU-WOD")
        print("="*60)

        results = {
            'mAP50-95': metrics.box.map50-95 if hasattr(metrics.box, 'map50-95') else 0,
            'mAP50': metrics.box.map50 if hasattr(metrics.box, 'map50') else 0,
            'mAP75': metrics.box.map75 if hasattr(metrics.box, 'map75') else 0,
            'Precision': metrics.box.mp if hasattr(metrics.box, 'mp') else 0,
            'Recall': metrics.box.mr if hasattr(metrics.box, 'mr') else 0,
        }

        # Calculate F1 score
        if results['Precision'] > 0 and results['Recall'] > 0:
            results['F1'] = 2 * (results['Precision'] * results['Recall']) / (results['Precision'] + results['Recall'])
        else:
            results['F1'] = 0

        for key, value in results.items():
            print(f"  {key}: {value:.4f}")

        # Per-class performance
        print("\n📊 Per-Class Performance:")
        if hasattr(metrics, 'box') and hasattr(metrics.box, 'ap_class_index'):
            for i, class_name in enumerate(CLASS_NAMES):
                if i < len(metrics.box.ap_class_index):
                    ap = metrics.box.ap_class_index[i]
                    print(f"  {class_name}: mAP50-95 = {ap:.4f}")

        return results

    except Exception as e:
        print(f"❌ Evaluation error: {e}")
        return None

metrics_results = evaluate_model()

# **STEP 11: OCCLUSION ANALYSIS**

In [ ]:
def advanced_occlusion_analysis(mask, bbox, image_shape):
    """
    Scientific occlusion analysis with multiple metrics
    Returns Occlusion Index (O.I.) from 0 to 1
    """
    try:
        mask_area = np.sum(mask)
        bbox_area = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
        area_ratio = mask_area / bbox_area if bbox_area > 0 else 0

        # Contour and Convex Hull analysis
        if mask_area > 0:
            contours = find_contours(mask, 0.5)
            if contours and len(contours) > 0:
                contour = max(contours, key=len)
                perimeter_len = perimeter(contour)
                hull = ConvexHull(contour)
                hull_area = hull.volume
                solidity = mask_area / hull_area if hull_area > 0 else 0
                complexity = perimeter_len / np.sqrt(mask_area) if mask_area > 0 else 0
            else:
                solidity = 0
                complexity = 0
        else:
            solidity = 0
            complexity = 0

        # Occlusion Index (O.I.)
        w1, w2, w3 = 0.5, 0.3, 0.2
        occlusion_index = 1 - (w1 * area_ratio + w2 * solidity + w3 * (1 - min(complexity/5, 1)))
        occlusion_index = np.clip(occlusion_index, 0, 1)

        if occlusion_index < 0.3:
            level = "low"
        elif occlusion_index < 0.6:
            level = "medium"
        else:
            level = "high"

        return {
            'index': float(occlusion_index),
            'level': level,
            'percentage': f"{occlusion_index*100:.1f}%",
            'metrics': {
                'area_ratio': float(area_ratio),
                'solidity': float(solidity),
                'complexity': float(min(complexity/5, 1))
            }
        }
    except Exception as e:
        print(f"⚠️ Occlusion analysis error: {e}")
        return {
            'index': 0.5,
            'level': 'medium',
            'percentage': '50.0%',
            'metrics': {'area_ratio': 0.5, 'solidity': 0.5, 'complexity': 0.5}
        }

print("✅ Occlusion analysis module ready")

# **STEP 12: LOAD FLORENCE**

In [ ]:
def load_florence():
    """Load Florence-2 with fixed compatibility"""

    try:
        clear_memory()
        print("\n📥 Loading Florence-2...")

        from transformers import AutoProcessor, AutoModelForCausalLM

        # Use stable version
        model_name = "microsoft/Florence-2-base"

        # Load processor
        processor = AutoProcessor.from_pretrained(
            model_name,
            trust_remote_code=True
        )

        # Load model with optimizations
        florence = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            trust_remote_code=True,
            low_cpu_mem_usage=True
        ).cuda()

        print("✅ Florence-2 loaded successfully!")
        return processor, florence

    except Exception as e:
        print(f"⚠️ Florence-2 loading error: {e}")
        print("📝 Continuing with YOLO-only mode...")
        return None, None

# Try to load Florence
processor, florence = load_florence()

# **STEP 13: TEXT GENERATOR**

In [ ]:
class SimpleTextGenerator:
    """Fallback text generator when Florence-2 is not available"""

    def generate_report(self, objects_info):
        if not objects_info:
            return "No objects detected in the image."

        class_counts = {}
        occlusion_stats = {'low': 0, 'medium': 0, 'high': 0}

        for obj in objects_info:
            cls = obj.get('class', 'Unknown')
            class_counts[cls] = class_counts.get(cls, 0) + 1
            occ_level = obj.get('occlusion', {}).get('level', 'medium')
            occlusion_stats[occ_level] = occlusion_stats.get(occ_level, 0) + 1

        report = "="*60 + "\n"
        report += "WASTE ANALYSIS REPORT\n"
        report += "="*60 + "\n\n"
        report += f"📊 Total Items Detected: {len(objects_info)}\n"
        report += f"📊 Materials Found: {', '.join(class_counts.keys())}\n\n"

        report += "📋 Composition:\n"
        for cls, count in class_counts.items():
            report += f"  • {cls}: {count} items ({count/len(objects_info)*100:.1f}%)\n"

        report += "\n📊 Occlusion Analysis:\n"
        report += f"  • Low Occlusion: {occlusion_stats['low']} items\n"
        report += f"  • Medium Occlusion: {occlusion_stats['medium']} items\n"
        report += f"  • High Occlusion: {occlusion_stats['high']} items\n"

        if objects_info:
            max_occ = max(objects_info, key=lambda x: x.get('occlusion', {}).get('index', 0))
            report += f"\n⚠️ Most Occluded Material: {max_occ.get('class', 'Unknown')} "
            report += f"({max_occ.get('occlusion', {}).get('percentage', 'N/A')})"

        report += "\n\n💡 Recommendations:\n"
        if occlusion_stats['high'] > len(objects_info) * 0.3:
            report += "  • High occlusion detected. Consider adjusting conveyor speed.\n"
        if 'Paper' in class_counts and class_counts['Paper'] > len(objects_info) * 0.4:
            report += "  • High paper content. Optimize pre-sorting stage.\n"
        report += "  • Regular maintenance recommended for optimal performance."

        return report

text_generator = SimpleTextGenerator()
print("✅ Text generator ready (fallback mode)")

# **STEP 14: VLM ANALYSIS FUNCTION**

In [ ]:
def analyze_with_vlm(image, yolo_results):
    """Analyze image using Florence-2 VLM (or fallback)"""

    try:
        # Extract YOLO results
        objects_info = []
        for r in yolo_results:
            if r.masks is not None:
                for box, mask, cls_id in zip(r.boxes, r.masks.data, r.boxes.cls):
                    bbox = box.xyxy[0].cpu().numpy()
                    cls_name = CLASS_NAMES[int(cls_id)]
                    conf = float(box.conf[0].cpu())

                    occl = advanced_occlusion_analysis(
                        mask.cpu().numpy(), bbox, image.size
                    )

                    objects_info.append({
                        'class': cls_name,
                        'confidence': conf,
                        'occlusion': occl,
                        'bbox': bbox.tolist()
                    })

        # Try Florence-2 if available
        if florence is not None and processor is not None:
            try:
                print("🔮 Generating VLM analysis with Florence-2...")

                # Task 1: Detailed Caption
                inputs = processor(
                    text="<DETAILED_CAPTION>",
                    images=image,
                    return_tensors="pt"
                ).cuda()

                with torch.no_grad():
                    generated_ids = florence.generate(
                        **inputs,
                        max_new_tokens=256,
                        num_beams=3,
                        temperature=0.7
                    )
                caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                # Task 2: Region Description
                prompt = f"""<REGION_TO_DESCRIPTION>
                Detected waste objects and occlusion levels:
                {json.dumps(objects_info, indent=2)}

                Provide professional analysis of the waste sorting line.
                """

                inputs = processor(text=prompt, images=image, return_tensors="pt").cuda()
                with torch.no_grad():
                    generated_ids = florence.generate(
                        **inputs,
                        max_new_tokens=512,
                        num_beams=3,
                        temperature=0.7
                    )
                analysis = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                # Task 3: VQA
                qa_prompt = "<VQA>\nQuestion: Which material category has the highest occlusion and why?"
                inputs = processor(text=qa_prompt, images=image, return_tensors="pt").cuda()
                with torch.no_grad():
                    generated_ids = florence.generate(
                        **inputs,
                        max_new_tokens=256,
                        num_beams=3
                    )
                vqa = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                return {
                    'caption': caption,
                    'analysis': analysis,
                    'vqa': vqa,
                    'objects': objects_info
                }

            except Exception as e:
                print(f"⚠️ Florence-2 generation error: {e}")
                print("🔄 Falling back to simple text generator...")

        # Fallback: Simple text generation
        analysis_text = text_generator.generate_report(objects_info)

        return {
            'caption': analysis_text[:200] + "..." if len(analysis_text) > 200 else analysis_text,
            'analysis': analysis_text,
            'vqa': "VQA not available (fallback mode)",
            'objects': objects_info
        }

    except Exception as e:
        print(f"❌ Analysis error: {e}")
        return {
            'caption': "Analysis error occurred",
            'analysis': "Error in analysis pipeline",
            'vqa': "Error in VQA",
            'objects': []
        }

print("✅ VLM analysis module ready")

# **STEP 15: VISUALIZATION**

In [ ]:
def visualize_results(image, results, save_path="analysis_result.png"):
    """
    Comprehensive visualization with:
    - Detection results with occlusion percentages
    - Class distribution chart
    - Statistical table
    - Analysis report
    """

    try:
        fig, axes = plt.subplots(2, 2, figsize=(20, 20))
        fig.suptitle(
            'Waste Analysis System: YOLOv8s-seg + Florence-2',
            fontsize=16,
            fontweight='bold'
        )

        # Subplot 1: Detection Visualization
        ax1 = axes[0, 0]
        ax1.imshow(image)

        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

        if results and 'objects' in results:
            for idx, obj in enumerate(results['objects'][:20]):
                bbox = obj.get('bbox', None)
                if bbox:
                    rect = plt.Rectangle(
                        (bbox[0], bbox[1]),
                        bbox[2]-bbox[0],
                        bbox[3]-bbox[1],
                        fill=False,
                        edgecolor=colors[idx % len(colors)],
                        linewidth=2
                    )
                    ax1.add_patch(rect)
                    ax1.text(
                        bbox[0],
                        bbox[1]-10,
                        f"{obj['class']} {obj.get('confidence',0):.2f} [{obj.get('occlusion', {}).get('percentage', 'N/A')}]",
                        color='white',
                        fontsize=8,
                        bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.7)
                    )

        ax1.set_title('Waste Detection with Occlusion Analysis')
        ax1.axis('off')

        # Subplot 2: Class Distribution
        ax2 = axes[0, 1]
        class_counts = {}
        if results and 'objects' in results:
            for obj in results['objects']:
                cls = obj.get('class', 'Unknown')
                class_counts[cls] = class_counts.get(cls, 0) + 1

        if class_counts:
            bars = ax2.bar(
                class_counts.keys(),
                class_counts.values(),
                color=colors[:len(class_counts)]
            )
            ax2.set_title('Waste Composition Distribution')
            ax2.set_xlabel('Material Type')
            ax2.set_ylabel('Count')
            ax2.grid(True, alpha=0.3)
            for bar, val in zip(bars, class_counts.values()):
                ax2.text(
                    bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.5,
                    str(val),
                    ha='center'
                )

        # Subplot 3: Statistical Table
        ax3 = axes[1, 0]
        ax3.axis('tight')
        ax3.axis('off')

        table_data = [['Material', 'Count', 'Avg Occlusion', 'Status']]

        if results and 'objects' in results and class_counts:
            for cls in class_counts.keys():
                occlusions = [
                    obj.get('occlusion', {}).get('index', 0)
                    for obj in results['objects']
                    if obj.get('class') == cls
                ]
                avg_occ = np.mean(occlusions) if occlusions else 0

                if avg_occ > 0.6:
                    status = '🔴 Critical'
                elif avg_occ > 0.3:
                    status = '🟡 Medium'
                else:
                    status = '🟢 Normal'

                table_data.append([
                    cls,
                    str(class_counts[cls]),
                    f"{avg_occ:.1%}",
                    status
                ])

        table = ax3.table(
            cellText=table_data,
            loc='center',
            cellLoc='center',
            colWidths=[0.25, 0.15, 0.25, 0.25]
        )
        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1.2, 1.5)
        ax3.set_title('Occlusion Analysis', fontsize=14, pad=20)

        # Subplot 4: Analysis Report
        ax4 = axes[1, 1]
        ax4.axis('off')

        report_text = f"""
        📊 INTELLIGENT ANALYSIS REPORT

        📝 Caption:
        {results.get('caption', 'No caption available')[:200]}...

        🔍 Key Findings:
        • Total Items: {len(results.get('objects', []))}
        • Most Occluded: {max(results.get('objects', []), key=lambda x: x.get('occlusion', {}).get('index', 0)).get('class', 'N/A') if results.get('objects') else 'N/A'}
        • Materials: {', '.join(class_counts.keys()) if class_counts else 'No objects detected'}

        💡 VLM Analysis:
        {results.get('analysis', 'No analysis available')[:300]}...
        """

        ax4.text(
            0.1, 0.9,
            report_text,
            fontsize=10,
            verticalalignment='top',
            bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.9)
        )
        ax4.set_title('VLM Analysis Report', fontsize=14, pad=20)

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

        print(f"✅ Visualization saved to: {save_path}")
        return save_path

    except Exception as e:
        print(f"❌ Visualization error: {e}")
        return None

print("✅ Visualization module ready")

# **STEP 16: TEST SYSTEM ON SAMPLES**

In [ ]:
def test_on_samples():
    """Test the complete system on sample images"""

    print("\n🔍 Testing system on sample images...")

    sample_dir = data_path / "2_Experiment_Sample" / "Merged_Purposed_Dataset"
    sample_images = list(sample_dir.glob("*.jpg")) + list(sample_dir.glob("*.png"))

    if not sample_images:
        print("⚠️ No sample images found. Using test images...")
        sample_images = list(test_img_dir.glob("*"))[:5]

    if not sample_images:
        print("❌ No images found!")
        return

    test_images = random.sample(sample_images, min(3, len(sample_images)))
    all_results = []

    for img_path in test_images:
        print(f"\n📸 Processing: {img_path.name}")

        image = Image.open(img_path)
        yolo_results = model(image, conf=0.25, iou=0.45)
        analysis_results = analyze_with_vlm(image, yolo_results)

        save_path = f"analysis_{img_path.stem}.png"
        visualize_results(image, analysis_results, save_path)

        all_results.append({
            'image': img_path.name,
            'objects': len(analysis_results.get('objects', [])),
            'classes': list(set(o.get('class', '') for o in analysis_results.get('objects', [])))
        })

    print("\n" + "="*60)
    print("TEST SUMMARY")
    print("="*60)
    for result in all_results:
        print(f"  {result['image']}: {result['objects']} objects, {', '.join(result['classes'])}")

    return all_results

test_summary = test_on_samples()

# **STEP 17: GENERATE FINAL REPORT**

In [ ]:
def generate_final_report():
    """Generate comprehensive JSON report"""

    try:
        report = {
            "project": {
                "name": "Vision-Language Waste Understanding under Occlusion",
                "version": "1.0.0",
                "date": str(pd.Timestamp.now()),
                "status": "Completed"
            },
            "dataset": {
                "name": "BUU Waste Occlusion Dataset (BUU-WOD)",
                "classes": CLASS_NAMES,
                "num_classes": NUM_CLASSES,
                "train_images": len(list(train_img_dir.glob('*'))),
                "valid_images": len(list(valid_img_dir.glob('*'))),
                "test_images": len(list(test_img_dir.glob('*')))
            },
            "models": {
                "detector": {
                    "name": "YOLOv8s-seg",
                    "type": "Instance Segmentation",
                    "parameters": "11.8M",
                    "size_mb": 23.9
                },
                "vlm": {
                    "name": "Florence-2-base",
                    "type": "Vision-Language Model",
                    "status": "Loaded" if florence is not None else "Fallback Mode"
                }
            },
            "performance": metrics_results if metrics_results else {},
            "test_results": test_summary,
            "technical_details": {
                "image_size": "640x640",
                "training_epochs": 30,
                "batch_size": 16,
                "augmentation": ["Mosaic", "MixUp", "Copy-Paste", "HSV", "Rotation", "Scale", "Shear"],
                "occlusion_metrics": ["Area Ratio", "Solidity", "Complexity", "Occlusion Index"]
            },
            "memory_usage": {
                "gpu_allocated_gb": torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0,
                "gpu_cached_gb": torch.cuda.memory_reserved() / 1e9 if torch.cuda.is_available() else 0
            }
        }

        with open('FINAL_PROJECT_REPORT.json', 'w') as f:
            json.dump(report, f, indent=2)

        print("\n✅ Final report saved to: FINAL_PROJECT_REPORT.json")
        return report

    except Exception as e:
        print(f"❌ Report generation error: {e}")
        return None

final_report = generate_final_report()

# **STEP 18: PROJECT SUMMARY**

In [ ]:

print("\n" + "="*70)
print("✅✅✅ PROJECT COMPLETED SUCCESSFULLY!")
print("="*70)

print("\n📊 PROJECT SUMMARY:")
print(f"  • Dataset: BUU-WOD with {NUM_CLASSES} classes")
print(f"  • Classes: {', '.join(CLASS_NAMES)}")
print(f"  • Model: YOLOv8s-seg (11.8M parameters)")
print(f"  • VLM: {'Florence-2' if florence is not None else 'Fallback Text Generator'}")

if metrics_results:
    print("\n📊 PERFORMANCE:")
    for key, value in metrics_results.items():
        print(f"  • {key}: {value:.4f}")

print("\n📁 OUTPUT FILES:")
print("  • FINAL_PROJECT_REPORT.json - Complete project report")
print("  • analysis_*.png - Detection visualizations")
print("  • analysis_result.png - Sample analysis")

if torch.cuda.is_available():
    print(f"\n💾 GPU MEMORY:")
    print(f"  • Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"  • Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

print("\n" + "="*70)
print("🎉 The VLM Waste Analysis System is ready for use!")
print("="*70)